# 06 XGBoost Notebook

## Load Data

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    precision_recall_curve,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)


In [2]:
DATA_PATH = Path("../data/processed/online_retail_II_labeled_30.csv")  # عدّلي المسار حسب اسم الملف عندك
df = pd.read_csv(DATA_PATH)
OUTPUT_DIR = Path("../data/processed")

df = df.sort_values(["CustomerID", "window_id"]).reset_index(drop=True)

print("Shape:", df.shape)
df.head()

Shape: (15582, 34)


,CustomerID,window_id,window_start,window_end,orders,spend,totalQuantity,unique_products,active_days,line_items,...,totalQuantity_change_pct,avargeOrderValue_change_pct,unique_products_change_pct,active_days_change_pct,items_per_order_change_pct,next_orders,next_spend,future_orders_change,future_spend_change,BehaviorShift
0,12346,6,2010-05-30,2010-06-29,1,142.31,19,19,1,19,...,2.800000,4.260998,2.800000,0.0,2.800000,1.0,77183.60,0.0,541.362448,0
1,12347,12,2010-11-26,2010-12-26,1,711.79,319,31,1,31,...,-0.373281,0.163949,-0.225000,0.0,-0.225000,1.0,475.39,0.0,-0.332120,1
2,12347,14,2011-01-25,2011-02-24,1,475.39,315,29,1,29,...,-0.012539,-0.332120,-0.064516,0.0,-0.064516,1.0,636.25,0.0,0.338375,0
3,12347,16,2011-03-26,2011-04-25,1,636.25,483,24,1,24,...,0.533333,0.338375,-0.172414,0.0,-0.172414,1.0,382.52,0.0,-0.398790,1
4,12347,18,2011-05-25,2011-06-24,1,382.52,196,18,1,18,...,-0.594203,-0.398790,-0.250000,0.0,-0.250000,1.0,584.91,0.0,0.529097,0


## 2- Drop leakage + identifier columns


In [3]:
TARGET_COL = "BehaviorShift"   

leakage_cols = [
    "next_orders", "next_spend",
    "future_orders_change", "future_spend_change",
]

other_target_cols = [
    c for c in df.columns
    if c.startswith("BehaviorShift") and c != TARGET_COL
]

identifier_cols = [
    "CustomerID", "window_id", "window_start", "window_end",
    "first_purchase", "last_purchase",
]

drop_cols = leakage_cols + other_target_cols + identifier_cols

X = df.drop(columns=drop_cols + [TARGET_COL], errors="ignore")
y = df[TARGET_COL]

print("Features used:", list(X.columns))
print("X shape:", X.shape, "| y shape:", y.shape)
print(y.value_counts(normalize=True))

Features used: ['orders', 'spend', 'totalQuantity', 'unique_products', 'active_days', 'line_items', 'avargeOrderValue', 'items_per_order', 'window_days', 'prev_orders', 'prev_spend', 'prev_totalQuantity', 'prev_avargeOrderValue', 'prev_unique_products', 'prev_active_days', 'prev_items_per_order', 'orders_change_pct', 'spend_change_pct', 'totalQuantity_change_pct', 'avargeOrderValue_change_pct', 'unique_products_change_pct', 'active_days_change_pct', 'items_per_order_change_pct']
X shape: (15582, 23) | y shape: (15582,)
BehaviorShift
0    0.647414
1    0.352586
Name: proportion, dtype: float64


## 3. Time-based train/test split

In [4]:
df_sorted = df.sort_values(["CustomerID", "window_id"]).reset_index(drop=True)

X = df_sorted.drop(columns=drop_cols + [TARGET_COL], errors="ignore")
y = df_sorted[TARGET_COL]

split_window = df_sorted["window_id"].quantile(0.8)
train_mask = df_sorted["window_id"] < split_window
test_mask = ~train_mask

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

## 4-Train the XGBoost model in train\test data

In [5]:
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

# X_train, X_test, y_train, y_test لازم تكون مبنية على TARGET_COL = BehaviorShift_30

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight:", scale_pos_weight)

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)

xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)
y_proba = xgb_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("PR-AUC:", average_precision_score(y_test, y_proba))

scale_pos_weight: 1.8362842032651183
              precision    recall  f1-score   support

           0       0.82      0.75      0.78      2102
           1       0.60      0.70      0.65      1145

    accuracy                           0.73      3247
   macro avg       0.71      0.73      0.72      3247
weighted avg       0.74      0.73      0.74      3247

[[1568  534]
 [ 338  807]]
ROC-AUC: 0.791834351979192
PR-AUC: 0.6606082369775408


## 5-Feature Importance

In [6]:
import pandas as pd

importances = pd.Series(
    xgb_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(importances.head(15))

orders                         0.659998
prev_active_days               0.043648
avargeOrderValue_change_pct    0.037769
spend                          0.022218
spend_change_pct               0.017253
line_items                     0.014897
avargeOrderValue               0.014622
prev_avargeOrderValue          0.014212
totalQuantity                  0.014002
prev_items_per_order           0.013904
prev_orders                    0.013652
prev_totalQuantity             0.013519
items_per_order_change_pct     0.013462
unique_products_change_pct     0.013274
prev_spend                     0.013247
dtype: float32


## 6- Hyperparameter Tuning with RandomizedSearchCV for XGBoost

In [7]:
from sklearn.model_selection import RandomizedSearchCV
import time

param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
}

xgb = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)

search = RandomizedSearchCV(
    xgb,
    param_distributions=param_dist,
    n_iter=30,
    scoring="average_precision",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

start = time.time()
search.fit(X_train, y_train)
print("Time taken:", round((time.time() - start)/60, 2), "minutes")
print("Best params:", search.best_params_)
print("Best CV score:", search.best_score_)

best_xgb = search.best_estimator_

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Time taken: 1.76 minutes
Best params: {'subsample': 1.0, 'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.01, 'colsample_bytree': 0.9}
Best CV score: 0.7081293994778317


## 7- Test the model after tuning at test set 

In [8]:
y_pred_final = best_xgb.predict(X_test)
y_proba_final = best_xgb.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_final))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_final))
print("PR-AUC:", average_precision_score(y_test, y_proba_final))

              precision    recall  f1-score   support

           0       0.84      0.75      0.79      2102
           1       0.61      0.73      0.67      1145

    accuracy                           0.74      3247
   macro avg       0.72      0.74      0.73      3247
weighted avg       0.76      0.74      0.74      3247

ROC-AUC: 0.8123837144079875
PR-AUC: 0.6893110251741664


## 8- Train the model without the orders column

In [9]:
X_train_no_orders = X_train.drop(columns=["orders"])
X_test_no_orders = X_test.drop(columns=["orders"])

xgb_model2 = XGBClassifier(**search.best_params_, scale_pos_weight=scale_pos_weight, eval_metric="logloss", random_state=42, n_jobs=-1)
xgb_model2.fit(X_train_no_orders, y_train)

y_pred2 = xgb_model2.predict(X_test_no_orders)
y_proba2 = xgb_model2.predict_proba(X_test_no_orders)[:, 1]

print(classification_report(y_test, y_pred2))
print("ROC-AUC:", roc_auc_score(y_test, y_proba2))
print("PR-AUC:", average_precision_score(y_test, y_proba2))

importances2 = pd.Series(xgb_model2.feature_importances_, index=X_train_no_orders.columns).sort_values(ascending=False)
print(importances2.head(15))

              precision    recall  f1-score   support

           0       0.83      0.74      0.79      2102
           1       0.61      0.72      0.66      1145

    accuracy                           0.74      3247
   macro avg       0.72      0.73      0.72      3247
weighted avg       0.75      0.74      0.74      3247

ROC-AUC: 0.8111588464303074
PR-AUC: 0.6836320406452061
orders_change_pct              0.427218
active_days                    0.139432
spend                          0.103380
avargeOrderValue_change_pct    0.043498
spend_change_pct               0.038496
avargeOrderValue               0.029110
line_items                     0.026191
prev_active_days               0.025782
prev_orders                    0.020099
active_days_change_pct         0.015778
prev_avargeOrderValue          0.015746
items_per_order_change_pct     0.013724
unique_products_change_pct     0.013244
prev_totalQuantity             0.012250
totalQuantity                  0.012232
dtype: float32


## 9- Hyperparameter Tuning with RandomizedSearchCV for XGBoost without Orders column

In [10]:
X_train_final = X_train.drop(columns=["orders"])
X_test_final = X_test.drop(columns=["orders"])

search_final = RandomizedSearchCV(
    XGBClassifier(scale_pos_weight=scale_pos_weight, eval_metric="logloss", random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=30,
    scoring="average_precision",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1,
)
search_final.fit(X_train_final, y_train)

print("Best params (final):", search_final.best_params_)
print("Best CV score (final):", search_final.best_score_)

final_model = search_final.best_estimator_

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best params (final): {'subsample': 1.0, 'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.01, 'colsample_bytree': 0.9}
Best CV score (final): 0.7027214623950483


## 10- Test the model after tuning at test set 

In [11]:
y_pred_final = final_model.predict(X_test_final)
y_proba_final = final_model.predict_proba(X_test_final)[:, 1]

print(classification_report(y_test, y_pred_final))
print(confusion_matrix(y_test, y_pred_final))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_final))
print("PR-AUC:", average_precision_score(y_test, y_proba_final))

              precision    recall  f1-score   support

           0       0.83      0.74      0.79      2102
           1       0.61      0.72      0.66      1145

    accuracy                           0.74      3247
   macro avg       0.72      0.73      0.72      3247
weighted avg       0.75      0.74      0.74      3247

[[1563  539]
 [ 316  829]]
ROC-AUC: 0.8111588464303074
PR-AUC: 0.6836320406452061


## 11- Saves the model

In [14]:
import os
import joblib

# Save final XGBoost model as version 1
model_version_dir = "../models/xgboost/v1"
os.makedirs(model_version_dir, exist_ok=True)

model_path = os.path.join(model_version_dir, "model.joblib")

joblib.dump(final_model, model_path)

print(f"Model saved at: {model_path}")

Model saved at: ../models/xgboost/v1\model.joblib


In [15]:
# Verify the saved model

loaded_model = joblib.load(model_path)

print("Model type:", type(loaded_model))
print("Number of features:", loaded_model.n_features_in_)
print("Model parameters:")
print(loaded_model.get_params())

Model type: <class 'xgboost.sklearn.XGBClassifier'>
Number of features: 22
Model parameters:
{'objective': 'binary:logistic', 'base_score': None, 'booster': None, 'callbacks': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': 0.9, 'device': None, 'early_stopping_rounds': None, 'enable_categorical': True, 'eval_metric': 'logloss', 'feature_types': None, 'feature_weights': None, 'gamma': None, 'grow_policy': None, 'importance_type': None, 'interaction_constraints': None, 'learning_rate': 0.01, 'max_bin': None, 'max_cat_threshold': None, 'max_cat_to_onehot': None, 'max_delta_step': None, 'max_depth': 6, 'max_leaves': None, 'min_child_weight': None, 'missing': nan, 'monotone_constraints': None, 'multi_strategy': None, 'n_estimators': 200, 'n_jobs': -1, 'num_parallel_tree': None, 'random_state': 42, 'reg_alpha': None, 'reg_lambda': None, 'sampling_method': None, 'scale_pos_weight': np.float64(1.8362842032651183), 'subsample': 1.0, 'tree_method': None, 'validate_

In [18]:
import json

model_metadata = {
    "model_name": "XGBoost",
    "version": "v1",
    "model_type": "XGBClassifier",

    "target": "BehaviorShift",

    "feature_set": {
        "name": "behavior_aware_without_orders",
        "n_features": 22,
        "features": [
            "spend",
            "totalQuantity",
            "unique_products",
            "active_days",
            "line_items",
            "avargeOrderValue",
            "items_per_order",
            "window_days",
            "prev_orders",
            "prev_spend",
            "prev_totalQuantity",
            "prev_avargeOrderValue",
            "prev_unique_products",
            "prev_active_days",
            "prev_items_per_order",
            "orders_change_pct",
            "spend_change_pct",
            "totalQuantity_change_pct",
            "avargeOrderValue_change_pct",
            "unique_products_change_pct",
            "active_days_change_pct",
            "items_per_order_change_pct"
        ]
    },

    "hyperparameters": {
        "n_estimators": int(final_model.n_estimators),
        "max_depth": int(final_model.max_depth),
        "learning_rate": float(final_model.learning_rate),
        "subsample": float(final_model.subsample),
        "colsample_bytree": float(final_model.colsample_bytree),
        "scale_pos_weight": float(final_model.scale_pos_weight),
        "random_state": int(final_model.random_state),
        "eval_metric": "logloss"
    },

    "test_metrics": {
        "accuracy": 0.74,
        "precision": 0.61,
        "recall": 0.72,
        "f1": 0.66,
        "roc_auc": 0.8111588464303074,
        "pr_auc": 0.6836320406452061
    },

    "test_confusion_matrix": [
        [1563, 539],
        [316, 829]
    ],

    "model_selection": {
        "final_model": "Tuned XGBoost without orders",
        "tuned_xgboost_evaluated": True,
        "orders_removal_experiment": True,
        "selection_metric": "PR-AUC",
        "reason": (
            "The tuned XGBoost model without the raw orders feature "
            "was selected because removing orders caused only a small "
            "performance decrease while producing a model more aligned "
            "with the project's objective of detecting behavioral shifts."
        )
    }
}

metadata_path = os.path.join(
    model_version_dir,
    "metadata.json"
)

with open(metadata_path, "w") as f:
    json.dump(model_metadata, f, indent=4)

print(f"Metadata saved at: {metadata_path}")

Metadata saved at: ../models/xgboost/v1\metadata.json
